[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Kernel_Methods.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Kernel Methods & RKHS

The elegant middle path between linear models and neural networks — and a proud UF lineage (information-theoretic learning and kernel adaptive filtering grew up here). Kernel trick, Gaussian processes with honest error bars, and KLMS: the [adaptive filter](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) gone nonlinear.

## 1. Pre-requisites

- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — inner products, projections (an RKHS is a Hilbert space with a bonus property).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2, [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 for the GP session.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def rbf(A, B, ell=0.5):
    """the Gaussian (RBF) kernel — similarity that decays with distance"""
    d2 = ((A[:, None, :] - B[None, :, :])**2).sum(-1)
    return np.exp(-d2 / (2 * ell**2))

---
### 🕐 Session 1 of 3 — *The Kernel Trick* (~35 min)
**Goal:** replace inner products with kernels; fit nonlinear functions with linear algebra.
**Builds on:** [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb). &nbsp; **Feeds into:** Session 2 (Gaussian processes).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Kernel Trick</b></summary>

**Timing (~35 min).** 10 min "linear methods only see inner products" · 10 min the representer theorem · 10 min kernel ridge regression · 5 min the lengthscale.

**Open with the observation that makes the whole trick possible, and let the room verify it.** Go through the linear methods they know — least squares, PCA, the LMS filter — and ask what the data actually enters through. In every case it is $x_i^Tx_j$, an inner product, and nothing else. **If an algorithm only ever touches inner products, you can swap in a different one.** That is the entire idea; everything after is checking that the swap is legal.

**Then make "legal" precise, because otherwise the trick sounds like cheating.** A kernel $k(x, x')$ is admissible when it *is* an inner product $\langle\phi(x), \phi(x')\rangle$ in some feature space. Mercer's condition says positive semi-definiteness is exactly the requirement. For the RBF kernel that feature space is **infinite-dimensional** — the expansion of $e^{x x'}$ has every polynomial degree — and you never construct it, never store it, never visit it. You evaluate a Gaussian bump. Rooms find this genuinely startling and it is worth letting the surprise land.

**The representer theorem is the load-bearing result, so give it a sentence and a consequence.** Minimising a regularised loss over an infinite-dimensional function space returns a solution of the form $f(x) = \sum_i \alpha_i k(x_i, x)$ — a weighted sum of kernels **centred on your data**. So the search over infinitely many functions collapses to solving for $n$ numbers. Say the consequence plainly: *your model is one bump per data point*, and that is both why the method is elegant and why Session 3's dictionary problem is unavoidable.

**Connect to the prerequisite rather than assuming it.** An RKHS is a [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) with one bonus property: evaluation at a point is itself an inner product, $f(x) = \langle f, k(x,\cdot)\rangle$. That reproducing property is what makes "the function value at $x$" a geometric operation, and it is the whole content of the R in RKHS.

**Read the code as three lines of linear algebra and say what each one is.** Build $K$; solve $(K + \lambda I)\alpha = y$; predict with $k(x_*, X)\alpha$. It is ridge regression with $K$ in place of $X^TX$. Note that $\lambda$ is not optional decoration — $K$ is often near-singular, and the ridge term is what makes the solve well-posed as well as what regularises. **One `np.linalg.solve` produces a nonlinear fit**, which is a fair summary of why kernel methods were attractive for two decades.

**Close on the lengthscale, and frame it as the model rather than as a hyperparameter.** $\ell$ sets how far influence spreads: small $\ell$ gives narrow bumps and a wiggly interpolant, large $\ell$ oversmooths. **It is the bias–variance dial compressed into one number**, and choosing it honestly — cross-validation, or marginal likelihood in Session 2 — is most of what kernel practice consists of. Worth flagging the target function's `0.5*sign(x)` kink: an RBF kernel assumes smoothness, so *no* lengthscale represents a discontinuity well. That misspecification is deliberate, and it is what the Session 2 calibration audit will detect.
</details>

## 2. Features Without Features

💡 **Intuition.** Linear methods only see inner products $x_i^T x_j$. The trick: replace every inner product with a **kernel** $k(x_i, x_j)$ — a similarity function that secretly equals an inner product in some (possibly infinite-dimensional) feature space. You get nonlinear power at linear-algebra prices, without ever visiting the feature space. The **representer theorem** seals it: the optimal function is always a weighted sum of kernels *centered on your data* — $f(x) = \sum_i \alpha_i k(x_i, x)$ — so the infinite-dimensional search collapses to solving for $n$ numbers. An RKHS is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) where evaluation *is* an inner product with a kernel bump.

In [2]:
# Kernel ridge regression from scratch: (K + λI)α = y — one linear solve, nonlinear fit
x = rng.uniform(-3, 3, 60)[:, None]
y = np.sin(2*x[:, 0]) + 0.5*np.sign(x[:, 0]) + 0.15*rng.standard_normal(60)   # kinked + noisy

lamr = 0.1
K = rbf(x, x)
alpha = np.linalg.solve(K + lamr*np.eye(len(x)), y)

xs = np.linspace(-3.4, 3.4, 400)[:, None]
f_hat = rbf(xs, x) @ alpha

plt.figure(figsize=(8, 2.8))
plt.plot(x, y, "k.", markersize=5, label="data")
plt.plot(xs, f_hat, label="kernel ridge (60 α's, one solve)")
plt.plot(xs, np.sin(2*xs) + 0.5*np.sign(xs), "k--", linewidth=0.8, label="truth")
plt.legend(fontsize=8); plt.title("nonlinear regression with nothing but linear algebra + a kernel")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2696075/3499237204.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** A curved, kinked function fitted by **one call to `np.linalg.solve`**. No iteration, no gradient descent, no learning rate, no epochs — build the $60\times60$ kernel matrix, solve $(K + \lambda I)\alpha = y$, and evaluate. Three lines, and the result tracks a target that no straight line could approach.

**Look at the shape of the code, because it is ridge regression with one substitution.** Ordinary ridge solves $(X^TX + \lambda I)w = X^Ty$ and predicts $x_*^Tw$. Here $K$ replaces $X^TX$ and $k(x_*, X)$ replaces $x_*^T$. **Every inner product was swapped for a kernel and nothing else changed** — that is the trick, stated operationally.

**And the feature space we are implicitly working in is infinite-dimensional.** The RBF kernel corresponds to a $\phi$ with infinitely many coordinates, so a naive "map the features then fit linearly" approach is not merely expensive, it is impossible. Yet the computation above is finite and small, because the representer theorem guarantees the optimum is $f(x) = \sum_i \alpha_i k(x_i, x)$ — **one bump per data point, 60 numbers**. The infinite search collapsed to a $60\times60$ solve.

**Note where the fit is honest and where it strains.** The smooth $\sin(2x)$ portion is tracked closely. The $0.5\,\mathrm{sign}(x)$ jump at the origin is **rounded off** — and it must be, because an RBF kernel encodes a prior that functions are infinitely differentiable, and no lengthscale makes a smooth kernel produce a discontinuity. This is deliberate model misspecification, planted so the calibration audit in Session 2 has something real to detect.

**The lengthscale is doing more work than the regulariser, and it is worth experimenting with here.** At $\ell = 0.5$ the bumps are narrow enough to follow the sine. Try $\ell = 0.1$: the curve interpolates every noisy point and looks like a comb. Try $\ell = 2$: the kink vanishes and the sine flattens. **One number spans underfitting to overfitting**, which makes kernel methods unusually easy to reason about and unusually sensitive to getting that number right. Session 2's marginal likelihood is the principled way to choose it.

**One caveat that becomes the central problem of Session 3.** This scales as $O(n^3)$ to solve and $O(n^2)$ to store, because $K$ is $n \times n$. At $n = 60$ that is instant; at $n = 10^5$ the matrix alone is 80 GB. **Kernel methods are exact and elegant and they do not scale**, and that single fact — not any deficiency of accuracy — is most of why neural networks displaced them once datasets grew.

**The lengthscale is the model.** $\ell$ controls how far influence spreads — small $\ell$ wiggles (overfits), large $\ell$ oversmooths. It's the bias-variance dial in one number, and choosing it honestly (cross-validation, or S2's marginal likelihood) is most of kernel practice.

---
### 🕐 Session 2 of 3 — *Gaussian Processes* (~40 min)
**Goal:** put a prior on functions; get predictions WITH calibrated uncertainty.
**Builds on:** Session 1; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3. &nbsp; **Feeds into:** Session 3 (kernel adaptive filters).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Gaussian Processes</b></summary>

**Timing (~40 min).** 10 min priors over functions · 12 min the posterior formulas and Cholesky · 10 min reading the error bars · 8 min the calibration audit.

**Sell the conceptual jump before any formula: a GP is [Bayesian estimation](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) with functions in place of parameters.** In that workshop you put a prior on $\theta$ and updated it with data. Here the prior is over **whole functions**, expressed through the kernel: "functions that are smooth with lengthscale $\ell$ are likely." Condition on data, and the posterior at any test point is a Gaussian with a closed-form mean and variance. No sampling, no variational approximation — a linear solve.

**Make the connection to Session 1 explicit, because it saves the room a lot of confusion.** The GP posterior *mean* is exactly kernel ridge regression, with $\lambda = \sigma_n^2$. Same numbers, same solve. **The GP adds one thing kernel ridge does not have: a variance.** Framing it that way means Session 2 is one new formula rather than a new method.

**Do the variance formula slowly, since its structure is the point.** $\mathrm{Var}(x_*) = k(x_*,x_*) - k_*^TK^{-1}k_*$: prior uncertainty *minus* what the data explains. The subtracted term is large when $x_*$ is close to observed points and vanishes when it is far away. So the error bars pinch at data and balloon in gaps **automatically**, from the algebra, with nothing added by hand.

**Note why the code uses Cholesky rather than `inv`.** $K$ is symmetric positive definite, so `cholesky` plus two triangular solves costs half of a general inverse and is far more stable. Forming $K^{-1}$ explicitly and multiplying is the textbook-to-code mistake; the numerical-linear-algebra lesson from [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S4 applies unchanged. Also flag `np.maximum(var, 0)`: the variance is mathematically non-negative but can go slightly negative in floating point when $x_*$ nearly coincides with a training point.

**Then the sentence the session exists for, and it is worth putting on the board.** A neural network extrapolates far outside its data with high confidence — the [ANN workshop](./Intro_ANN/Intro_ANN.ipynb) boundary plot shows exactly that. A GP widens its band and **tells you it does not know**. When data is scarce and the cost of a confident wrong answer is high, that property is worth more than accuracy.

**Make the calibration audit the centrepiece, because most GP tutorials stop before it.** Drawing a pretty ±2σ band proves nothing; the question is whether the band is *right*. 500 fresh points, count how many fall inside, compare against 95%. **This is a falsifiable test of an uncertainty claim**, and running it is the difference between a plot and a measurement.

**Be ready to interpret the 92%, which is close to but not equal to 95%.** Undercoverage of three points is not rounding — it is the kernel's smoothness prior meeting a target with a genuine discontinuity at the origin. The GP is confident near $x = 0$ because it has data on both sides and expects smoothness, but the truth jumps by 1.0 there, so the residuals near the kink are much larger than the model believes. **A misspecified prior gives wrong error bars, and this audit detected it.** That is a far more valuable lesson than a demo that returned exactly 95%.

**If time allows, name the escape routes.** Use a Matérn-1/2 kernel, which is far less smooth and would handle the kink; or learn $\ell$ and $\sigma_n$ by maximising the marginal likelihood rather than fixing them; or accept the misspecification and inflate $\sigma_n$. Each is a real practice, and each is a response to a diagnostic the room just ran themselves.
</details>

## 3. Distributions Over Functions

💡 **Intuition.** A GP is [Bayesian estimation](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) upgraded from parameters to *whole functions*: the kernel plays the prior ('smooth functions with lengthscale ℓ are likely'), data updates it, and the posterior at any test point is a Gaussian — mean **and variance**, in closed form. The error bars behave the way honesty demands: pinched near data, ballooning in the gaps. Where a neural net extrapolates with confidence it hasn't earned, a GP *tells you it doesn't know*.

In [3]:
sn = 0.15                                        # known noise level
Kxx = rbf(x, x) + sn**2*np.eye(len(x))
Ks  = rbf(xs, x)
Kss = rbf(xs, xs)

L = np.linalg.cholesky(Kxx)
alpha_gp = np.linalg.solve(L.T, np.linalg.solve(L, y))
mu = Ks @ alpha_gp
v = np.linalg.solve(L, Ks.T)
var = np.diag(Kss) - (v**2).sum(0)
sd = np.sqrt(np.maximum(var, 0))

plt.figure(figsize=(8, 2.8))
plt.plot(x, y, "k.", markersize=5)
plt.plot(xs, mu, label="GP posterior mean")
plt.fill_between(xs[:, 0], mu-2*sd, mu+2*sd, alpha=0.2, label="±2σ")
plt.legend(fontsize=8); plt.title("uncertainty pinches at data, balloons in the gaps — as it should")
plt.tight_layout(); plt.show()

# calibration audit on fresh data
x_new = rng.uniform(-3, 3, 500)[:, None]
y_new = np.sin(2*x_new[:, 0]) + 0.5*np.sign(x_new[:, 0]) + 0.15*rng.standard_normal(500)
mu_new = rbf(x_new, x) @ alpha_gp
v2 = np.linalg.solve(L, rbf(x, x_new))
sd_new = np.sqrt(np.maximum(1 - (v2**2).sum(0) + sn**2, 0))
inside = np.mean(np.abs(y_new - mu_new) <= 2*sd_new)
print(f"fresh points inside the ±2σ band: {inside:.0%}  (well-calibrated ≈ 95%)")

fresh points inside the ±2σ band: 92%  (well-calibrated ≈ 95%)


/tmp/ipykernel_2696075/1619739755.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two things — a posterior with error bars, and a **test of whether those error bars are honest**. The band pinches where data is dense and widens in the gaps, exactly as the formula demands. And on 500 fresh points, **92%** fall inside the ±2σ band against a nominal 95%.

**The shape of the band is not a stylistic choice; it falls out of the algebra.** The posterior variance is $k(x_*,x_*) - k_*^TK^{-1}k_*$ — prior uncertainty *minus* what the data explains. Near an observation the subtracted term is large and the band collapses toward the noise level; far from any observation it vanishes and the variance returns to the prior. **Nothing was tuned to make the picture look right.**

**Note also what the posterior mean is.** It is *identical* to the kernel ridge fit from Session 1, with $\lambda = \sigma_n^2$. Same solve, same numbers. The GP's entire addition is the second moment — which is why it costs nothing extra to have.

**Now the audit, which is the part most GP tutorials omit.** A pretty ±2σ band proves nothing; the claim it makes is falsifiable, so falsify it. Draw fresh data, count the fraction inside the band, compare against 95%. **This turns an uncertainty claim into a measurement**, and it is the habit worth taking from this cell.

**And the result is 92%, which is undercoverage — the band is slightly too narrow, and the reason is diagnosable.** Three percentage points on 500 points is about 15 examples, against a binomial standard error of $\sqrt{0.95 \times 0.05/500} \approx 1\%$, so this is roughly **three standard errors low**: a real effect, not sampling noise. The cause is the target's `0.5*sign(x)` discontinuity. The RBF kernel encodes a prior that functions are infinitely differentiable, so near $x = 0$ the GP sees data on both sides, assumes a smooth transition, and reports *high confidence* — while the truth jumps by 1.0. Residuals near the kink are far larger than the model believes.

**So the audit did its job: it detected a misspecified prior.** That is a better outcome than a demo returning exactly 95%. **Calibration is a property of the model–data pair, not of Bayesian machinery in the abstract** — a GP gives you *coherent* uncertainty given its assumptions, and coherent is not the same as correct. Get the kernel wrong and you get confidently wrong error bars, with all the elegance intact.

**Three ways to fix it, each a real practice.** Switch to a Matérn-1/2 kernel, whose sample paths are non-differentiable and which handles kinks comfortably. Learn $\ell$ and $\sigma_n$ by maximising the marginal likelihood instead of fixing them by hand — the principled version of Session 1's lengthscale question. Or accept the misspecification and inflate $\sigma_n$ to buy coverage, which is honest about the width but not about the cause.

**Still, keep the comparison that motivates the session in view.** The [ANN workshop](./Intro_ANN/Intro_ANN.ipynb) boundary plot showed a network confidently colouring regions it had never seen. This GP is imperfectly calibrated at 92% and **widens its band in every gap**, which is a categorically different failure from unearned confidence. When data is scarce and a confident wrong answer is expensive, that difference is worth more than accuracy.

---
### 🕐 Session 3 of 3 — *Kernel Adaptive Filters: KLMS* (~40 min)
**Goal:** run LMS in the RKHS: a nonlinear adaptive filter, one sample at a time.
**Builds on:** Session 1; [APA workshop](../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Kernel Adaptive Filters (KLMS)</b></summary>

**Timing (~40 min).** 8 min the batch-to-online jump · 10 min deriving KLMS from LMS · 12 min the experiment · 10 min the dictionary problem.

**Frame the jump precisely.** Sessions 1 and 2 were **batch**: all the data present, one big solve, $O(n^3)$. Real filters do not get that — samples arrive one at a time and a decision is needed now. This session runs the same RKHS idea **online**, and the result is the [LMS filter](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with its linearity assumption removed. Worth noting the local lineage: kernel adaptive filtering and information-theoretic learning grew up at UF, and this workshop is a direct descendant.

**Derive KLMS rather than presenting it, because the derivation is two lines and it demystifies the method completely.** LMS is $w_n = w_{n-1} + \mu e_n x_n$. Write the same update in the RKHS, where the "weight vector" is a function and the "input vector" is $k(x_n, \cdot)$:

$$f_n = f_{n-1} + \mu e_n \, k(x_n, \cdot)$$

Unroll it from $f_0 = 0$ and you get $f_n = \sum_{i \le n} \mu e_i k(x_i, \cdot)$ — the representer theorem arriving **automatically**, not as a theorem you had to invoke. Say this out loud: KLMS never needed the representer theorem, it constructs the representation as a side effect of the update.

**Give the physical reading.** Each sample plants a Gaussian bump at its own location, with height proportional to the error it caused. Samples the filter already predicts well plant nothing. **The filter is a landscape of bumps built where it was surprised**, which is a memorable and accurate description.

**Set up the experiment as a fair fight with a predetermined loser.** The system is $\tanh(1.5 \times \text{FIR}(u))$ — a linear filter followed by a saturating nonlinearity. A linear filter *cannot* represent it at any tap count, so linear LMS has a floor set by the nonlinearity, not by noise. Ask the room to predict what more taps would buy the linear filter: nothing. **This is a representational limit, not a convergence one**, and that distinction is the point of the session.

**Read the result honestly, because the gap is real but the winner has not converged either.** Steady-state MSE is 0.0931 linear against 0.0288 for KLMS — a genuine 5.1 dB win. But the additive noise is $0.05\sigma$, so the achievable floor is $0.0025$, and KLMS sits **11× above it**. The right summary is "the kernel buys what no linear width can", not "KLMS solved the problem." Both statements are visible in the same two numbers, and only one of them is on the plot's title.

**Then the cost, which is the honest counterweight and the reason the session ends where it does.** By sample 2000 the dictionary holds **2000 centres** — one per sample, never pruned. Prediction at step $t$ costs $O(t)$, so total cost is $O(N^2)$, memory grows without bound, and the filter is unusable in a real-time loop. Ask what an engineer does about it, and let the room propose merging nearby centres before you name QKLMS.

**Close by connecting the dictionary problem to the workshop's parting claim.** Kernel methods are exact, elegant, uncertainty-aware, and **linear in memory with the data they have seen**. Neural networks are approximate and fixed-size. That trade is the whole story of why one displaced the other as datasets grew — and why kernels still win when data is scarce and error bars matter.
</details>

## 4. LMS Meets the Kernel

💡 **Intuition.** [LMS](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) updates a weight vector; **KLMS** runs the identical update *in the RKHS* — by the representer theorem the filter is a growing sum of kernel bumps, one planted on each sample, weighted by $\mu e_n$: $f_n = f_{n-1} + \mu e_n \, k(x_n, \cdot)$. It learns *nonlinear* systems online with LMS's simplicity. The price is the growing dictionary — practical variants (QKLMS) merge nearby bumps to cap it.

In [4]:
# nonlinear system id: y = tanh of a filtered input — LMS can't, KLMS can
from scipy import signal as sig
N = 2000
u = rng.standard_normal(N)
lin = sig.lfilter([1, 0.5, -0.3], [1], u)
d = np.tanh(1.5*lin) + 0.05*rng.standard_normal(N)

L_emb = 5                                          # embed the last 5 inputs as the "x"
U = np.stack([np.roll(u, k) for k in range(L_emb)], 1); U[:L_emb] = 0

def lms_lin(U, d, mu=0.05):
    w = np.zeros(U.shape[1]); e = np.zeros(len(d))
    for t in range(len(d)):
        e[t] = d[t] - w @ U[t]
        w += mu * e[t] * U[t]
    return e

def klms(U, d, mu=0.5, ell=1.0):
    centers, coefs = [], []
    e = np.zeros(len(d))
    for t in range(len(d)):
        if centers:
            kv = np.exp(-((np.array(centers) - U[t])**2).sum(1) / (2*ell**2))
            y = np.dot(coefs, kv)
        else:
            y = 0.0
        e[t] = d[t] - y
        centers.append(U[t].copy()); coefs.append(mu * e[t])
    return e

e_lin, e_k = lms_lin(U, d), klms(U, d)
def curve(e): return 10*np.log10(np.convolve(e**2, np.ones(80)/80, "valid") + 1e-12)
plt.figure(figsize=(8, 2.8))
plt.plot(curve(e_lin), label="linear LMS: floored by the nonlinearity")
plt.plot(curve(e_k), label="KLMS: keeps descending")
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.xlabel("sample"); plt.ylabel("MSE [dB]")
plt.title("nonlinear system: the kernel buys what no linear width can")
plt.tight_layout(); plt.show()
print(f"steady-state MSE  linear {np.mean(e_lin[-500:]**2):.4f}   KLMS {np.mean(e_k[-500:]**2):.4f}")

steady-state MSE  linear 0.0931   KLMS 0.0288


/tmp/ipykernel_2696075/2130777877.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Steady-state MSE **0.0931 for linear LMS against 0.0288 for KLMS** — a factor of 3.2, or **5.1 dB**. The learning curves show the mechanism: linear LMS descends and then **flattens onto a floor**, while KLMS keeps going.

**The linear filter's floor is a representational limit, not a convergence one — and that distinction is the whole session.** The system is $\tanh(1.5 \times \mathrm{FIR}(u))$: a linear filter followed by a saturating nonlinearity. No linear filter can produce a saturating output, at any tap count, with any step size, given any amount of data. **More taps buy nothing.** The residual is the part of the nonlinearity the model cannot express, and the curve flattening is that impossibility made visible.

**KLMS clears the floor because its hypothesis class contains the target.** The update $f_n = f_{n-1} + \mu e_n k(x_n, \cdot)$ is LMS written in the RKHS, and unrolling it from $f_0 = 0$ gives $f_n = \sum_i \mu e_i k(x_i, \cdot)$ — the representer theorem appearing **for free**, as a consequence of the update rather than as a theorem invoked. Each sample plants a Gaussian bump at its own location with height proportional to the error it caused. The filter is a landscape of bumps built wherever it was surprised.

**Now the number the plot title does not mention: KLMS has not converged either.** The additive noise is $0.05\sigma$, so the achievable MSE floor is $0.05^2 = 0.0025$. KLMS sits at 0.0288 — **11× above it**. So the honest reading is that the kernel buys what no linear width can, *and* that 2000 samples with $\mu = 0.5$, $\ell = 1$ is not enough to finish the job. Both facts live in the same two numbers.

**And the cost is the counterweight this session exists to deliver.** The dictionary grows by **one centre per sample, forever** — 2000 centres by the end, none pruned. Prediction at step $t$ costs $O(t)$, so the run is $O(N^2)$ in time and $O(N)$ in memory, both unbounded. Look at the inner loop: `np.array(centers)` rebuilds the entire dictionary on every single sample, which is why this is a teaching implementation and not a deployable filter. **An adaptive filter whose cost grows with uptime cannot run in a real-time loop.**

**Which is exactly what the practical variants attack.** QKLMS quantises the input space and *merges* a new sample into an existing centre when it falls within a threshold, capping the dictionary at a size set by the input distribution rather than by the sample count. Novelty criteria and coherence-based sparsification do the same job by different tests. All of them answer one question: **which of these bumps do I actually need?**

**Zoom out and the trade is the story of the field.** Kernel methods are exact, uncertainty-aware, and **grow with the data they have seen**. Neural networks are approximate and fixed-size. When data is scarce and error bars matter, the first wins — as Session 2 showed. When data is enormous, the dictionary problem is fatal and the second wins. Understanding *why* networks won is worth as much as knowing that they did.

## 5. Conclusion

Kernels buy nonlinearity at linear-algebra prices; GPs add honest uncertainty; KLMS carries it all online. When data is scarce and error bars matter, this toolbox still beats deep learning — and when data is huge, you'll understand *why* networks won (the dictionary problem).

---
## Where next

- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — chasing GP-quality error bars with deep models.
- [RLS workshop](../Intro_Time_Series/Intro_RLS.ipynb) — KRLS: the recursive version.
- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — the geometry under all of it.